In [1]:
# %%
!rm -rf ./*
!zip -r anonymized.zip ./anonymized

"rm" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"zip" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ============================================================
# КОНСТАНТЫ МОДУЛЯ
# ============================================================
DIRECTORY = '/content/'

MONTHS_RU = {
    'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
    'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
    'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
}

NUMERIC_COLS = [
    # Поры
    'Поры - Filter [mm]', 'Поры - Maximum Area [mm²]', 'Поры - Threshold [mm]',
    'Поры - Общий объём [mm³]', 'Поры - Средний объём пор [mm³]',
    'Поры - Общая площадь пор [mm²]', 'Поры - Средняя площадь пор [mm²]',
    'Поры - Максимальная глубина [mm]', 'Поры - Плотность [cm⁻²]',
    'Поры - Количество', 'Поры - Индекс',
    # Складки
    'Складки - Filter [mm]', 'Складки - Minimum Length [mm]', 'Складки - Threshold [mm]',
    'Складки - Длина [mm]', 'Складки - Total area [mm²]', 'Складки - Общий объём [mm³]',
    'Складки - Средняя ширина [mm]', 'Складки - Средняя глубина [mm]',
    'Складки - Максимальная глубина [mm]', 'Складки - Итоговое число',
    'Складки - Индекс углубления',
    # Тонкие морщины
    'Тонкие морщины - Filter [mm]', 'Тонкие морщины - Minimum Length [mm]',
    'Тонкие морщины - Threshold [mm]', 'Тонкие морщины - Длина [mm]',
    'Тонкие морщины - Total area [mm²]', 'Тонкие морщины - Общий объём [mm³]',
    'Тонкие морщины - Оценка', 'Тонкие морщины - Средняя ширина [mm]',
    'Тонкие морщины - Средняя глубина [mm]', 'Тонкие морщины - Максимальная глубина [mm]',
    'Тонкие морщины - Итоговое число', 'Тонкие морщины - Индекс углубления',
]

def clean_numeric_cols(df, cols=NUMERIC_COLS):
    """Преобразует указанные колонки в float, заменяя запятые на точки.
    Нечисловые значения становятся NaN. Колонки, которых нет в df, игнорируются."""
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )
    return df

def parse_ru_date(s):
    try:
        parts = str(s).split()
        return datetime(int(parts[2]), MONTHS_RU[parts[1]], int(parts[0])).date()
    except:
        return None

def extract_region(s):
    if pd.isna(s):
        return None
    parts = str(s).split(' - ', 1)
    return parts[1]

def join_regions(region):
    if pd.isna(region):
        return region
    r = str(region).lower()
    if ('лоб' in r) or ('лба' in r):
        return 'Лоб'
    if 'гусиные лапки' in r:
        return 'Гусиные лапки'
    if 'носогубная складка' in r:
        return 'Носогубная складка'
    if 'скула' in r:
        return 'Скула'
    if 'щека' in r:
        return 'Щека'
    return region

def load_and_clean(directory=DIRECTORY):
    """Загрузка + очистка всех CSV. Создаёт колонки date, stage, region."""
    all_dfs = []
    for filename in sorted(os.listdir(directory)):
        f = os.path.join(directory, filename)
        if not os.path.isfile(f) or not filename.endswith('.csv'):
            continue
        df = pd.read_csv(f, sep=';')
        df['patient'] = filename[:-4]
        df = clean_numeric_cols(df)

        df['date'] = df.iloc[:, 2].apply(parse_ru_date)

        unique_dates = sorted(df['date'].dropna().unique())
        if len(unique_dates) >= 2:
            df['stage'] = (df['date'] != unique_dates[0]).astype(int)
        else:
            df['stage'] = pd.NA

        df['region_raw'] = df['Image context'].apply(extract_region)
        df['region'] = df['region_raw'].apply(join_regions)
        all_dfs.append(df)
    return pd.concat(all_dfs, ignore_index=True)

def task1_first_look(filename='З.Утконосова.csv'):
    df = pd.read_csv(os.path.join(DIRECTORY, filename), sep=';')
    print("shape:", df.shape)
    print("\ndtypes:\n", df.dtypes)
    print("\nhead:\n", df.head(3))
    print("\nПропуски по колонкам (топ-10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    print("\nУникальные Image context:", df['Image context'].nunique())
    print(df['Image context'].unique())

    return df